<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/stage_07b_model_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07b_- SEQ2ONE - Model Implementation**

In [1]:
window_sizes = [180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']
n_features = 36

# **Bloque genérico de ejecución**

## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [4]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [5]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [6]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [7]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [9]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [10]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y

In [11]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [12]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [13]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [14]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
    ):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [18]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**


In [19]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

## **9. Gestión de dataset de métricas**

In [20]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/models/seq2one_metrics_final",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [21]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/models/seq2one_metrics_final",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [22]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **IMPLEMENTACIÓN DE MODELOS**

## **10. Implementación de modelo Transformer**

Implementación del modelo Transformer seq2one para `delta_60` con `L=180` con: **Selección final de hiperparámetros**

| cfg_id | R2_valid_mean | R2_valid_std | R2_test_mean | R2_test_std | DA_valid_mean | DA_test_mean | MAE_valid_mean | MAE_test_mean | gap_valid_minus_test_mean |
|-------:|--------------:|-------------:|-------------:|------------:|--------------:|-------------:|---------------:|--------------:|---------------------------:|
| 2 | 0.586100 | 0.009491 | 0.476808 | 0.021605 | 0.791358 | 0.784349 | 18.050566 | 30.365389 | 0.109291 |

Se adopta formalmente la siguiente configuración `cfg_id = 2`:

- Hiperparametros de arquitectura:
     - `d_model = 64`
     - `nhead = 8`
     - `num_layers = 2`
     - `dim_ff = 512`
     - `pooling = "last"`

- Hiperparametros de aprendizaje:
   - `dropout ≈ 0.0847`
   - `lr ≈ 2.46e-4`
   - `weight_decay ≈ 2.2e-5`

### **10.1. Imports y “seed” (base reproducible)**

In [23]:
# Paso 1: imports básicos + reproducibilidad (sin tqdm)
import os
import json
import random
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [24]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # reproducibilidad (puede bajar performance, pero estable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### **10.2. TensorDataset + DataLoader (PyTorch)**

In [25]:
# Generador de Dataloader para tuning con variación por seed.
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_loaders_from_bundle_3d(
    bundle: dict,
    *,
    seq_len: int | None = None,
    n_features: int | None = None,
    batch_size: int = 1024,
    num_workers: int = 2,
) -> dict:
    """
    Crea loaders train/valid/test para TRANSFORMER many-to-one.

    Espera:
      - X: (n, seq_len, n_features)  (ya 3D)
      - y: (n,) o (n,1)  -> (n,1)

    Si seq_len/n_features se pasan, valida consistencia.
    """
    loaders = {}


    import random

    g = torch.Generator()
    g.manual_seed(42)

    def _seed_worker(worker_id):
        worker_seed = 42 + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 3:
            raise ValueError(
                f"[{split}] Se esperaba X 3D (n, seq_len, n_features). "
                f"Recibido shape={X.shape} (ndim={X.ndim})."
            )

        n, sl, nf = X.shape

        if seq_len is not None and sl != int(seq_len):
            raise ValueError(f"[{split}] seq_len esperado={seq_len}, recibido={sl}. shape={X.shape}")

        if n_features is not None and nf != int(n_features):
            raise ValueError(f"[{split}] n_features esperado={n_features}, recibido={nf}. shape={X.shape}")

        if y.shape[0] != n:
            raise ValueError(f"[{split}] X e y no alinean: X n={n}, y n={y.shape[0]}.")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")
        #
        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
            generator=g if shuffle else None,
            worker_init_fn=_seed_worker if num_workers > 0 else None,
        )

    return loaders

### **10.3. Definición de modelo Transformer (many-to-one)**

In [26]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """
    Positional Encoding sinusoidal (Vaswani et al.).
    Asume entradas con forma (B, L, D) y agrega información de posición.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # pe: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)

        # div_term: (d_model/2,) para índices pares
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # pares
        pe[:, 1::2] = torch.cos(position * div_term)  # impares

        # Guardamos como buffer para que:
        # - se mueva con .to(device)
        # - no sea parámetro entrenable
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, d_model)
        L = x.size(1)
        x = x + self.pe[:, :L, :]
        return self.dropout(x)


class TransformerManyToOne(nn.Module):
    """
    Transformer Encoder many-to-one para regresión:
      - Entrada:  (B, L, F)
      - Salida:   (B, 1)

    Sugerencias integradas:
      - Validación d_model % nhead == 0 (evita trials inválidos en tuning).
      - LayerNorm post-proyección de entrada (a menudo estabiliza/regulariza).
      - Pooling configurable ("mean" o "last") para reducir (B, L, D) -> (B, D).

    Nota:
      - No incluye máscaras de padding porque asume L fijo (sin padding).
        Si más adelante usa padding, habrá que pasar src_key_padding_mask al encoder.
    """
    def __init__(
        self,
        *,
        n_features: int,
        d_model: int = 64,
        nhead: int = 8,
        num_layers: int = 2,
        dim_ff: int = 512,
        dropout: float = 0.0847,
        pooling: str = "last",
        max_len: int = 2048,
    ):
        super().__init__()

        # 1) Validación clave para Optuna/grid: evita combinaciones inválidas
        if d_model % nhead != 0:
            raise ValueError(f"d_model ({d_model}) debe ser múltiplo de nhead ({nhead}).")

        if pooling not in ("mean", "last"):
            raise ValueError(f"pooling inválido: {pooling}. Use 'mean' o 'last'.")

        self.pooling = pooling

        # Proyección de features a dimensión del modelo
        self.in_proj = nn.Linear(n_features, d_model)

        # 2) (Opcional pero recomendado) normalización para estabilizar la entrada
        self.in_norm = nn.LayerNorm(d_model)

        # Positional encoding
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout, max_len=max_len)

        # Encoder layers
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # entrada/salida (B, L, D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Cabeza de regresión
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, F)
        z = self.in_proj(x)   # (B, L, D)
        z = self.in_norm(z)   # (B, L, D)  -> estabiliza escalas
        z = self.pos_enc(z)   # (B, L, D)
        z = self.encoder(z)   # (B, L, D)

        # Pooling: reduce dimensión temporal
        if self.pooling == "last":
            pooled = z[:, -1, :]     # (B, D)
        else:
            pooled = z.mean(dim=1)   # (B, D)

        out = self.head(pooled)      # (B, 1)
        return out

In [27]:
# ============================================================
# 2) Modelo: factory configurable para tuning
# ============================================================
import torch.nn as nn

def make_transformer_model(
    *,
    n_features: int,
    device: torch.device,
    d_model: int = 64,
    nhead: int = 8,
    num_layers: int = 2,
    dim_ff: int = 512,
    dropout: float = 0.0847,
    pooling: str = "last",
) -> nn.Module:
    """
    Factory del Transformer many-to-one.

    Permite variar hiperparámetros desde Optuna o grid search.
    """

    model = TransformerManyToOne(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
    ).to(device)

    return model

In [28]:
# ------------------------------------------------------------
# Smoke test robusto (por split/horizonte)
# ------------------------------------------------------------
def smoke_test(model, loader, device, name: str):
    """
    Verifica rápidamente:
      - shapes esperados: X (B,L,F), y (B,) o (B,1), out (B,1)
      - dtypes
      - alineación batch entre y y out
      - loader no vacío
      - NaNs/Infs en X, y, out
    """
    if loader is None:
        raise ValueError(f"[{name}] Loader es None.")

    model.eval()

    it = iter(loader)
    try:
        xb, yb = next(it)
    except StopIteration:
        raise ValueError(f"[{name}] Loader vacío: no hay batches para validar.")

    xb = xb.to(device)
    yb = yb.to(device)

    with torch.no_grad():
        out = model(xb)

    print(f"[{name}] X:", tuple(xb.shape), xb.dtype)
    print(f"[{name}] y:", tuple(yb.shape), yb.dtype)
    print(f"[{name}] out:", tuple(out.shape), out.dtype)

    # --- checks estructurales ---
    assert xb.ndim == 3, f"[{name}] X debe ser 3D: (B, L, F). Recibido {xb.shape}"
    assert out.ndim == 2 and out.shape[1] == 1, f"[{name}] Salida debe ser (B, 1). Recibido {out.shape}"
    assert yb.ndim in (1, 2), f"[{name}] y debe ser (B,) o (B,1). Recibido {yb.shape}"

    # --- alineación batch ---
    B = xb.shape[0]
    assert out.shape[0] == B, f"[{name}] Batch mismatch: X B={B}, out B={out.shape[0]}"

    if yb.ndim == 1:
        assert yb.shape[0] == B, f"[{name}] Batch mismatch: y B={yb.shape[0]}, esperado {B}"
    else:  # yb.ndim == 2
        assert yb.shape == (B, 1), f"[{name}] y 2D debe ser (B,1). Recibido {yb.shape}"

    # --- NaNs/Infs ---
    assert torch.isfinite(xb).all(), f"[{name}] X contiene NaN/Inf"
    assert torch.isfinite(yb).all(), f"[{name}] y contiene NaN/Inf"
    assert torch.isfinite(out).all(), f"[{name}] out contiene NaN/Inf"

### **10.4. Definición de loss, optimizer y funciones de train / eval (sin tqdm)**

In [29]:
# ============================================================
# loss, optimizer y funciones de entrenamiento / evaluación
# + hiperparámetros FINALES (Transformer cfg_id=2)
# ============================================================

import torch
import torch.nn as nn
from typing import Optional


# ------------------------------------------------------------
# HP finales (Transformer cfg_id=2)
# ------------------------------------------------------------
FINAL_LR = 2.46e-4
FINAL_WEIGHT_DECAY = 2.2e-5
FINAL_DROPOUT = 0.0847  # (se usa al instanciar el modelo, no aquí)


# ------------------------------------------------------------
# Factory: Loss (regresión)
# ------------------------------------------------------------
def make_criterion() -> nn.Module:
    # MSE para entrenamiento (estable y estándar en regresión)
    return nn.MSELoss()


# ------------------------------------------------------------
# Factory: Optimizer (uno por corrida/modelo)
# ------------------------------------------------------------
def make_optimizer(
    model: nn.Module,
    *,
    lr: float = FINAL_LR,
    weight_decay: float = FINAL_WEIGHT_DECAY,
) -> torch.optim.Optimizer:
    # AdamW suele funcionar bien con Transformers
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


# ------------------------------------------------------------
# Helper: AMP scaler (crear 1 vez y reutilizar)
# ------------------------------------------------------------
def make_amp_scaler(device: torch.device, *, use_amp: bool = True) -> torch.amp.GradScaler:
    amp_enabled = bool(use_amp and device.type == "cuda")
    return torch.amp.GradScaler("cuda", enabled=amp_enabled)


# ------------------------------------------------------------
# Función de entrenamiento (1 epoch)
# ------------------------------------------------------------
def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    *,
    scaler: Optional[torch.amp.GradScaler] = None,
    clip_grad_norm: Optional[float] = None,
    use_amp: bool = True,   # AMP en CUDA
) -> float:
    model.train()
    total_loss = 0.0
    n_samples = 0

    amp_enabled = bool(use_amp and device.type == "cuda")
    if scaler is None:
        scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Normalización segura de shape de y -> (B,1)
        if yb.ndim == 1:
            yb = yb.unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)

        # Forward + loss (AMP opcional)
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            y_hat = model(xb)
            loss = criterion(y_hat, yb)

        # Backward + step (AMP opcional)
        scaler.scale(loss).backward()

        if clip_grad_norm is not None and clip_grad_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Función de evaluación (1 epoch)
# ------------------------------------------------------------
@torch.no_grad()
def eval_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
    *,
    use_amp: bool = True,
) -> float:
    model.eval()
    total_loss = 0.0
    n_samples = 0

    amp_enabled = bool(use_amp and device.type == "cuda")

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Normalización segura de shape de y -> (B,1)
        if yb.ndim == 1:
            yb = yb.unsqueeze(1)

        with torch.amp.autocast("cuda", enabled=amp_enabled):
            y_hat = model(xb)
            loss = criterion(y_hat, yb)

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Ejemplo de uso (hiperparámetros finales ya aplicados)
# ------------------------------------------------------------
# criterion = make_criterion()
# optimizer = make_optimizer(model)  # usa FINAL_LR y FINAL_WEIGHT_DECAY por default
# scaler = make_amp_scaler(device)
#
# train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler=scaler)
# valid_loss = eval_one_epoch(model, valid_loader, criterion, device)
#
# Nota: Para seleccionar HP por VALID, calcule R²/DA por fuera (usando preds completas).

### **10.5. Loop de entrenamiento completo con early stopping**



In [30]:
import math
import torch
import torch.nn as nn
from typing import Optional, Dict, Any

# ------------------------------------------------------------
# HP finales (Transformer cfg_id=2)
# ------------------------------------------------------------
FINAL_LR = 2.46e-4
FINAL_WEIGHT_DECAY = 2.2e-5


def fit_one_run(
    *,
    model: nn.Module,
    train_loader,
    valid_loader,
    device: torch.device,
    lr: float = FINAL_LR,
    weight_decay: float = FINAL_WEIGHT_DECAY,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    save_best: bool = True,
    best_path: Optional[str] = None,
    clip_grad_norm: Optional[float] = None,
    use_amp: bool = True,          # (opcional) si usa AMP en train/eval
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Entrena 1 modelo (1 target/horizonte/window_size) con early stopping en VALID.

    - Optimiza MSE (loss).
    - Early stopping monitorea valid_loss.
    - Si save_best=True, restaura el mejor estado al final.
    """
    criterion = make_criterion()
    optimizer = make_optimizer(model, lr=lr, weight_decay=weight_decay)

    # (Recomendado) AMP scaler persistente durante toda la corrida
    scaler = make_amp_scaler(device, use_amp=use_amp)

    best_val = math.inf
    best_epoch = -1
    patience_left = patience

    history = {"train_loss": [], "valid_loss": []}
    best_state = None

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            scaler=scaler,
            clip_grad_norm=clip_grad_norm,
            use_amp=use_amp,
        )
        val_loss = eval_one_epoch(
            model,
            valid_loader,
            criterion,
            device,
            use_amp=use_amp,
        )

        history["train_loss"].append(float(train_loss))
        history["valid_loss"].append(float(val_loss))

        # mejora real si supera min_delta
        improved = (best_val - val_loss) > min_delta

        if improved:
            best_val = float(val_loss)
            best_epoch = epoch
            patience_left = patience

            if save_best:
                # Copia segura del state_dict a CPU (para restaurar luego sin depender del device)
                best_state = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}

                # Guardado opcional a disco
                if best_path is not None:
                    torch.save(model.state_dict(), best_path)
        else:
            patience_left -= 1

        if verbose:
            print(
                f"epoch {epoch:02d} | "
                f"train_loss={train_loss:.6f} | "
                f"valid_loss={val_loss:.6f} | "
                f"patience_left={patience_left}"
            )

        if patience_left <= 0:
            if verbose:
                print(f"Early stopping: best_valid_loss={best_val:.6f} at epoch {best_epoch}")
            break

    # Restaurar mejor modelo
    if save_best and best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
        if verbose:
            print(f"Modelo restaurado: epoch {best_epoch} | best_valid_loss={best_val:.6f}")

    return {
        "best_valid_loss": best_val,
        "best_epoch": best_epoch,
        "epochs_ran": epoch,   # último epoch ejecutado (incluye early stop)
        "history": history,
    }

### **10.6. Predicciones Transformer**


Función de predicción (seq2one) -> y_true, y_pred

In [31]:
import numpy as np
import torch


@torch.no_grad()
def predict_seq2one(
    model,
    loader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Genera predicciones completas para un loader seq2one.

    Retorna:
        y_true: shape (N,)
        y_pred: shape (N,)

    Normaliza automáticamente shapes (B,) o (B,1).
    """

    model.eval()

    y_true_list = []
    y_pred_list = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Forward
        y_hat = model(xb)

        # CPU numpy
        y_pred = y_hat.detach().cpu().numpy().astype(np.float32)
        y_true = yb.detach().cpu().numpy().astype(np.float32)

        # Normalizar shapes -> (B,)
        if y_pred.ndim == 2 and y_pred.shape[1] == 1:
            y_pred = y_pred.squeeze(1)

        if y_true.ndim == 2 and y_true.shape[1] == 1:
            y_true = y_true.squeeze(1)

        y_pred_list.append(y_pred)
        y_true_list.append(y_true)

    if len(y_true_list) == 0:
        raise ValueError("Loader vacío en predict_seq2one")

    y_true_all = np.concatenate(y_true_list, axis=0)
    y_pred_all = np.concatenate(y_pred_list, axis=0)

    return y_true_all, y_pred_all

##Ejemplo de uso:
##y_true, y_pred = predict_seq2one(model, valid_loader, device)

In [32]:
import numpy as np

def get_metrics_torch_from_loaders(
    loaders: dict,
    model,
    *,
    device: torch.device,
    predict_loader_fn=predict_seq2one,
    compute_r2: bool = True,
) -> tuple[dict, dict]:
    """
    Calcula métricas VALID y TEST para modelos seq2one usando DataLoaders.

    Requisitos:
      - loaders debe contener keys: 'valid' y 'test'
      - predict_loader_fn(model, loader, device=...) -> (y_true_all, y_pred_all)
      - compute_seq2one_metrics(y_true, y_pred, ...) debe devolver dict con métricas (MAE/RMSE/R2/DA, etc.)
    """

    # Validaciones defensivas (errores más claros)
    if "valid" not in loaders or "test" not in loaders:
        raise KeyError("loaders debe contener las claves 'valid' y 'test'.")

    # -------- VALID --------
    y_true_valid, y_pred_valid = predict_loader_fn(model, loaders["valid"], device=device)
    y_true_valid = np.asarray(y_true_valid).reshape(-1)
    y_pred_valid = np.asarray(y_pred_valid).reshape(-1)

    metrics_valid = compute_seq2one_metrics(y_true_valid, y_pred_valid, compute_r2=compute_r2)

    # -------- TEST --------
    y_true_test, y_pred_test = predict_loader_fn(model, loaders["test"], device=device)
    y_true_test = np.asarray(y_true_test).reshape(-1)
    y_pred_test = np.asarray(y_pred_test).reshape(-1)

    metrics_test = compute_seq2one_metrics(y_true_test, y_pred_test, compute_r2=compute_r2)

    return metrics_valid, metrics_test

### **10.7. Función `train_transformer`**

In [33]:
from typing import Any, Dict, Tuple
import torch
import torch.nn as nn
L = 180
models_dir = "/content/drive/MyDrive/neural_profit/final_models"
best_path = f"{models_dir}/transformer_L{L}_{[targets][0]}.pt"

def train_transformer(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,
    d_model: int = 64,
    nhead: int = 8,
    num_layers: int = 2,
    dim_ff: int = 512,
    dropout: float = 0.0847,
    pooling: str = "last",
    lr: float = 2.46e-4,
    weight_decay: float = 2.2e-5,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    use_scheduler: bool = False,
    save_best: bool = True,
    best_path: str | None = None,
    use_amp: bool = True,
    verbose: bool = True,
) -> Tuple[nn.Module, Dict[str, Any], Dict, Dict]:
    """
    Entrena 1 Transformer (1 target/horizonte/window_size) y devuelve:
      (model, hist, metrics_valid, metrics_test)

    Nota:
      - La seed NO se fija acá. Debe fijarse afuera.
    """

    # 0) Validaciones defensivas
    for k in ("train", "valid", "test"):
        if k not in loaders:
            raise KeyError(f"loaders debe contener '{k}'.")

    # 1) Modelo
    model = TransformerManyToOne(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
    ).to(device)

    # 2) Fit (early stopping en VALID)
    hist = fit_one_run(
        model=model,
        train_loader=loaders["train"],
        valid_loader=loaders["valid"],
        device=device,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        patience=patience,
        min_delta=min_delta,
        save_best=save_best,
        best_path=best_path,
        clip_grad_norm=clip_grad_norm,
        use_amp=use_amp,
        verbose=verbose,
    )

    # 3) Métricas VALID / TEST
    metrics_valid, metrics_test = get_metrics_torch_from_loaders(
        {"valid": loaders["valid"], "test": loaders["test"]},
        model,
        device=device,
        predict_loader_fn=predict_seq2one,
        compute_r2=True,
    )

    return model, hist, metrics_valid, metrics_test

### **10.8. Implementación final**

In [45]:
from pathlib import Path
import json
import time
import gc
import torch

# ============================================================
# 0) Configuración final
# ============================================================
target = "delta_60"
window_size = 180
n_features = 36

batch_size_train = 4096

# HP finales Transformer
hp_model = {
    "d_model": 64,
    "nhead": 8,
    "num_layers": 2,
    "dim_ff": 512,
    "dropout": 0.0847,
    "pooling": "last",
}

# HP finales optimización
hp_optim = {
    "lr": 2.46e-4,
    "weight_decay": 2.2e-5,
    "max_epochs": 30,
    "patience": 5,
    "min_delta": 0.0,
    "clip_grad_norm": 1.0,
    "use_amp": True,
}

models_dir = Path("/content/drive/MyDrive/neural_profit/final_models")
models_dir.mkdir(parents=True, exist_ok=True)

best_path = models_dir / f"transformer_L{window_size}_{target}.pt"
summary_path = best_path.with_name(best_path.stem + "_summary.json")

print("best_path:", str(best_path))
print("summary_path:", str(summary_path))


# ============================================================
# 1) Verificar si el modelo y el resumen ya existen
# ============================================================
model_exists = best_path.exists()
summary_exists = summary_path.exists()

print("\n=== Verificación de archivos ===")
print("Modelo existe :", model_exists)
print("Resumen existe:", summary_exists)

transformer_model = None
transformer_model_summary = None

# ------------------------------------------------------------
# Caso A: si ambos archivos existen, cargarlos y no entrenar
# ------------------------------------------------------------
if model_exists and summary_exists:
    print("\n[INFO] Ya existen archivos para esta configuración.")
    print("Cargando modelo desde:")
    print(best_path)
    print("Cargando resumen desde:")
    print(summary_path)

    # reconstruir arquitectura
    transformer_model = TransformerManyToOne(
        n_features=n_features,
        **hp_model
    ).to(device)

    # cargar pesos
    state_dict = torch.load(best_path, map_location=device)
    transformer_model.load_state_dict(state_dict)
    transformer_model.eval()

    # cargar json resumen
    with open(summary_path, "r", encoding="utf-8") as f:
        transformer_model_summary = json.load(f)

    print("\n[OK] Modelo y resumen cargados correctamente.")

# ------------------------------------------------------------
# Caso B: si no existen ambos, entrenar y guardar
# ------------------------------------------------------------
else:
    print("\n[INFO] No se encontraron ambos archivos. Se procederá a entrenar.")

    # ============================================================
    # 2) Crear bundle
    # ============================================================
    (bundle,) = create_bundles(
        window_size=window_size,
        targets=[target],
        windows_paths=windows_paths,
        scalers_paths=scalers_paths,
        flatten_X=False,   # Transformer necesita input 3D: (N, L, F)
    )

    print("Bundle creado:")
    print("train X:", bundle["train"]["X"].shape, "| y:", bundle["train"]["y"].shape)
    print("valid X:", bundle["valid"]["X"].shape, "| y:", bundle["valid"]["y"].shape)
    print("test  X:", bundle["test"]["X"].shape,  "| y:", bundle["test"]["y"].shape)

    # ============================================================
    # 3) Crear DataLoaders
    # ============================================================
    loaders = make_loaders_from_bundle_3d(
        bundle,
        seq_len=window_size,
        n_features=n_features,
        batch_size=batch_size_train,
    )

    print("\nLoaders creados:")
    print("n_train:", len(loaders["train"].dataset))
    print("n_valid:", len(loaders["valid"].dataset))
    print("n_test :", len(loaders["test"].dataset))

    # ============================================================
    # 4) Smoke test opcional
    # ============================================================
    smoke_test(
        model=TransformerManyToOne(
            n_features=n_features,
            **hp_model
        ).to(device),
        loader=loaders["train"],
        device=device,
        name="train",
    )

    # ============================================================
    # 5) Entrenar Transformer final + guardar checkpoint
    # ============================================================
    t0 = time.perf_counter()

    transformer_model, hist, metrics_valid, metrics_test = train_transformer(
        loaders=loaders,
        n_features=n_features,
        device=device,

        # HP finales Transformer
        **hp_model,

        # HP finales optimización
        **hp_optim,

        # guardado del mejor modelo
        save_best=True,
        best_path=str(best_path),

        verbose=True,
    )

    dt_train = time.perf_counter() - t0

    # ============================================================
    # 6) Guardar JSON resumen
    # ============================================================
    transformer_model_summary = {
        "model_name": "transformer_seq2one",
        "target": target,
        "window_size": window_size,
        "n_features": n_features,
        "batch_size_train": batch_size_train,
        "hp_model": hp_model,
        "hp_optim": hp_optim,
        "hist": hist,
        "metrics_valid": metrics_valid,
        "metrics_test": metrics_test,
        "paths": {
            "model_path": str(best_path),
            "summary_path": str(summary_path),
        },
        "timing": {
            "train_seconds": float(dt_train),
        },
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(transformer_model_summary, f, ensure_ascii=False, indent=2)

    print("\nResumen guardado en:")
    print(str(summary_path))

    print("\n=== HIST ===")
    print(hist)

    print("\n=== METRICS VALID ===")
    print(metrics_valid)

    print("\n=== METRICS TEST ===")
    print(metrics_test)

    print("\nModelo guardado en:")
    print(str(best_path))

    # liberar memoria auxiliar
    bundle = None
    loaders = None
    hist = None
    metrics_valid = None
    metrics_test = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n[CLEAN] Memoria liberada.")


# ============================================================
# 7) Variables finales esperadas
# ============================================================
print("\n=== OBJETOS FINALES ===")
print("transformer_model:", type(transformer_model))
print("transformer_model_summary:", type(transformer_model_summary))

best_path: /content/drive/MyDrive/neural_profit/final_models/transformer_L180_delta_60.pt
summary_path: /content/drive/MyDrive/neural_profit/final_models/transformer_L180_delta_60_summary.json

=== Verificación de archivos ===
Modelo existe : True
Resumen existe: True

[INFO] Ya existen archivos para esta configuración.
Cargando modelo desde:
/content/drive/MyDrive/neural_profit/final_models/transformer_L180_delta_60.pt
Cargando resumen desde:
/content/drive/MyDrive/neural_profit/final_models/transformer_L180_delta_60_summary.json


/tmp/ipykernel_827/586612775.py:95: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)



[OK] Modelo y resumen cargados correctamente.

=== OBJETOS FINALES ===
transformer_model: <class '__main__.TransformerManyToOne'>
transformer_model_summary: <class 'dict'>


### **10.9. Conclusiones — Entrenamiento final del Transformer (L = 180, target = delta_60)**

**1. Desempeño en VALID**

Las métricas obtenidas en el conjunto de validación son:

- **MAE ≈ 18.01**
- **RMSE ≈ 31.05**
- **Directional Accuracy (DA) ≈ 0.791**
- **R² ≈ 0.576**

Estos resultados muestran un desempeño sólido del modelo en validación y son **consistentes con los resultados observados durante el análisis multi-seed realizado en el proceso de tuning**.  
En particular, el valor de **DA cercano a 0.79** indica una alta capacidad del modelo para predecir correctamente la dirección del movimiento del mercado.

---

**2. Desempeño en TEST (fuera de muestra)**

Las métricas obtenidas en el conjunto de test son:

- **MAE ≈ 29.74**
- **RMSE ≈ 60.10**
- **Directional Accuracy (DA) ≈ 0.789**
- **R² ≈ 0.471**

En el conjunto fuera de muestra el modelo mantiene:

- **Directional Accuracy muy alta**
- **R² sólido**
- **MAE competitivo**

Esto indica que el modelo **generaliza adecuadamente** y que la configuración seleccionada mantiene un buen desempeño fuera de muestra.

---

**3. Comportamiento del entrenamiento**

Durante el entrenamiento se obtuvo:

- **best_epoch = 11**
- **best_valid_loss = 964.25**

El proceso de *early stopping* detuvo el entrenamiento en:

- **epoch = 16**

Este comportamiento es consistente con un entrenamiento estable: el modelo mejora progresivamente hasta alcanzar un óptimo en validación y posteriormente comienza a oscilar, por lo que el *early stopping* evita continuar el entrenamiento y reduce el riesgo de sobreajuste.

---

**4. Confirmación de la arquitectura seleccionada**

Los resultados obtenidos confirman que:

- La **arquitectura seleccionada del Transformer** funciona correctamente en la corrida final.
- El comportamiento del modelo es **consistente con los resultados obtenidos durante el proceso de tuning y análisis multi-seed**.
- El modelo entrenado y guardado en: `/content/drive/MyDrive/neural_profit/final_models/transformer_L180_delta_60.pt`


## **11. Implementación de modelo MLP**

La configuración seleccionada es:

**cfg_id = 1**

| cfg_id | R2_valid_mean | R2_valid_std | R2_test_mean | R2_test_std | DA_valid_mean | DA_test_mean | MAE_valid_mean | MAE_test_mean | gap_valid_minus_test_mean |
|-------:|--------------:|-------------:|-------------:|------------:|--------------:|-------------:|---------------:|--------------:|---------------------------:|
| 1 | 0.458434 | 0.012114 | 0.457211 | 0.015802 | 0.746978 | 0.728173 | 21.431548 | 35.983137 | 0.001223 |

**Hiperparámetros finales**

- `hidden_dims` = `[512, 256, 128]`
- `activation` = `relu`
- `dropout` ≈ `0.187`
- `lr` ≈ `5.13e-4`
- `weight_decay` ≈ `6.6e-5`
- `window_size` = `180`
- `target` = `delta_60`

In [35]:
# ==========================================
# 1) Configuración final MLP
# ==========================================
FINAL_WINDOW_SIZE = 180
FINAL_TARGET = "delta_60"

FINAL_MLP_CONFIG = {
    "hidden_dims": [512, 256, 128],
    "activation": "relu",
    "dropout": 0.187,
    "lr": 5.13e-4,
    "weight_decay": 6.6e-5,
    "batch_size": 4096,
}

### **11.1. Make Loaders 2D**

In [36]:
def make_loaders_from_bundle(bundle, *, batch_size: int = 4096, num_workers: int = 0) -> dict:
    loaders = {}
    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 2:
            raise ValueError(f"[{split}] X debe ser 2D (n, d). Got shape={X.shape}")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"[{split}] X y y deben tener mismo n. X={X.shape}, y={y.shape}")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )
    return loaders

### **11.2. Definición del modelo MLP (simple y controlado)**

In [37]:
import torch
import torch.nn as nn

def _get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    if name == "silu" or name == "swish":
        return nn.SiLU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"activation no soportada: {name}")

class MLPSeq2One(nn.Module):
    """
    MLP configurable para seq2one:
      - in_dim: dimensión de entrada (d)
      - hidden_dims: lista de dimensiones ocultas
      - activation: relu | gelu | silu | tanh
      - dropout: float
      - norm: none | layernorm | batchnorm
      - residual: aplica skip en bloques con misma dim (opcional)
    """
    def __init__(
        self,
        *,
        in_dim: int,
        hidden_dims: list[int],
        activation: str = "relu",
        dropout: float = 0.0,
        norm: str = "none",
        residual: bool = False,
    ):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims debe tener al menos 1 capa")

        norm = norm.lower()
        dims = [in_dim] + list(hidden_dims)
        layers: list[nn.Module] = []

        for i in range(len(dims) - 1):
            d_in, d_out = dims[i], dims[i + 1]
            block: list[nn.Module] = [nn.Linear(d_in, d_out)]

            if norm == "layernorm":
                block.append(nn.LayerNorm(d_out))
            elif norm == "batchnorm":
                block.append(nn.BatchNorm1d(d_out))
            elif norm == "none":
                pass
            else:
                raise ValueError(f"norm no soportada: {norm}")

            block.append(_get_activation(activation))

            if dropout and dropout > 0:
                block.append(nn.Dropout(dropout))

            layers.append(nn.Sequential(*block))

        self.blocks = nn.ModuleList(layers)
        self.residual = residual
        self.out = nn.Linear(dims[-1], 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2:
            x = x.reshape(x.size(0), -1)

        h = x
        for block in self.blocks:
            h_new = block(h)
            if self.residual and (h_new.shape[-1] == h.shape[-1]):
                h = h + h_new
            else:
                h = h_new

        return self.out(h)

### **11.3. Entrenamiento con early stopping (VALID)**

In [38]:
from datetime import datetime

def _ts():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    sse = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        sse += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return sse / max(n, 1)

def train_mlp(
    loaders: dict,
    *,
    model_cfg: dict,      # <-- arquitectura
    optim_cfg: dict,      # <-- hiperparams aprendizaje
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,
):
    """
    Entrena MLP seq2one con early stopping en VALID por MSE.
    Retorna: (model_best, best_valid_mse, epochs_ran)
    """
    if verbose:
        print(
            f"[{_ts()}] [TRAIN] START | model_cfg={model_cfg} | "
            f"optim_cfg={optim_cfg} | max_epochs={max_epochs} patience={patience} | device={device.type}"
        )

    t_global = time.perf_counter()

    model = MLPSeq2One(**model_cfg).to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=float(optim_cfg.get("lr", 1e-3)),
        weight_decay=float(optim_cfg.get("weight_decay", 0.0)),
    )
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0
    epochs_ran = 0

    for epoch in range(1, max_epochs + 1):
        epochs_ran = epoch
        t_epoch = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={train_loss_sum/max(train_n,1):.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        t0 = time.perf_counter()
        valid_mse = evaluate_mse(model, loaders["valid"], device)
        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | epochs_ran={epochs_ran} | dt_total={dt_all:.2f}s")

    return model, best_valid, epochs_ran

### **11.4. Predicciones MLP**


In [39]:
@torch.no_grad()
def predict_mlp(model, X: np.ndarray, *, device: torch.device, batch_size: int = 32768) -> np.ndarray:
    """
    Predice con un modelo MLP PyTorch en batches.
    Retorna shape (n_samples,)
    """
    model.eval()

    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 2:
        raise ValueError(f"X debe ser 2D (n, d). Got shape={X.shape}")

    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).reshape(-1)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)

In [40]:
def get_metrics_torch(bundle, model, *, device) -> tuple[dict, dict]:

    # -------- VALID --------
    X_valid = bundle["valid"]["X"]
    y_valid = np.asarray(bundle["valid"]["y"]).reshape(-1)
    y_pred_valid = np.asarray(
        predict_mlp(model, X_valid, device=device)
    ).reshape(-1)

    metrics_valid = compute_seq2one_metrics(
        y_valid, y_pred_valid, compute_r2=True
    )

    # -------- TEST --------
    X_test = bundle["test"]["X"]
    y_test = np.asarray(bundle["test"]["y"]).reshape(-1)
    y_pred_test = np.asarray(
        predict_mlp(model, X_test, device=device)
    ).reshape(-1)

    metrics_test = compute_seq2one_metrics(
        y_test, y_pred_test, compute_r2=True
    )

    return metrics_valid, metrics_test

### **11.5. Función `train_mlp`**


In [41]:
from typing import Any, Dict, Tuple
from pathlib import Path

import torch
import torch.nn as nn

def train_mlp(
    bundle: dict,
    loaders: dict,
    *,
    in_dim: int,
    device: torch.device,
    hidden_dims: list[int] = [512, 256, 128],
    activation: str = "relu",
    dropout: float = 0.187,
    norm: str = "none",
    residual: bool = False,
    lr: float = 5.13e-4,
    weight_decay: float = 6.6e-5,
    max_epochs: int = 30,
    patience: int = 5,
    save_best: bool = True,
    best_path: str | None = None,
    verbose: bool = True,
) -> Tuple[nn.Module, Dict[str, Any], Dict, Dict]:
    """
    Entrena 1 MLP seq2one (1 target / 1 horizon / 1 window_size) y devuelve:
      (model, hist, metrics_valid, metrics_test)

    Notas:
      - La seed NO se fija acá. Debe fijarse afuera.
      - Early stopping sobre VALID usando MSE.
      - Si save_best=True y best_path no es None, guarda el mejor state_dict.
      - Las métricas finales de VALID/TEST se calculan desde `bundle`
        usando `predict_mlp(...)` vía `get_metrics_torch(...)`.
    """

    # --------------------------------------------------
    # 0) Validaciones defensivas
    # --------------------------------------------------
    for k in ("train", "valid", "test"):
        if k not in loaders:
            raise KeyError(f"loaders debe contener '{k}'.")
        if k not in bundle:
            raise KeyError(f"bundle debe contener '{k}'.")

    if in_dim <= 0:
        raise ValueError(f"in_dim debe ser > 0. Recibido: {in_dim}")

    # --------------------------------------------------
    # 1) Modelo
    # --------------------------------------------------
    model = MLPSeq2One(
        in_dim=in_dim,
        hidden_dims=hidden_dims,
        activation=activation,
        dropout=dropout,
        norm=norm,
        residual=residual,
    ).to(device)

    # --------------------------------------------------
    # 2) Optimizador / loss
    # --------------------------------------------------
    opt = torch.optim.Adam(
        model.parameters(),
        lr=float(lr),
        weight_decay=float(weight_decay),
    )
    loss_fn = nn.MSELoss()

    # --------------------------------------------------
    # 3) Entrenamiento + early stopping
    # --------------------------------------------------
    hist = {
        "train_loss": [],
        "valid_mse": [],
        "best_valid_mse": None,
        "best_epoch": None,
        "epochs_ran": 0,
        "best_path": best_path,
        "model_name": "MLPSeq2One",
        "model_cfg": {
            "in_dim": in_dim,
            "hidden_dims": hidden_dims,
            "activation": activation,
            "dropout": dropout,
            "norm": norm,
            "residual": residual,
        },
        "optim_cfg": {
            "lr": lr,
            "weight_decay": weight_decay,
            "max_epochs": max_epochs,
            "patience": patience,
        },
    }

    best_state = None
    best_valid_mse = float("inf")
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for xb, yb in loaders["train"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

        train_loss_avg = train_loss_sum / max(train_n, 1)
        valid_mse = evaluate_mse(model, loaders["valid"], device)

        hist["train_loss"].append(float(train_loss_avg))
        hist["valid_mse"].append(float(valid_mse))
        hist["epochs_ran"] = epoch

        improved = valid_mse < best_valid_mse

        if improved:
            best_valid_mse = float(valid_mse)
            best_epoch = epoch
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            if save_best and best_path is not None:
                best_path_obj = Path(best_path)
                best_path_obj.parent.mkdir(parents=True, exist_ok=True)
                torch.save(best_state, best_path_obj)

        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"epoch={epoch:02d} | "
                f"train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag}"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[EARLY STOPPING] best_valid_mse={best_valid_mse:.6f} en epoch={best_epoch}")
            break

    # --------------------------------------------------
    # 4) Restaurar mejor modelo
    # --------------------------------------------------
    if best_state is not None:
        model.load_state_dict(best_state)

    hist["best_valid_mse"] = best_valid_mse
    hist["best_epoch"] = best_epoch

    # --------------------------------------------------
    # 5) Métricas VALID / TEST
    # --------------------------------------------------
    metrics_valid, metrics_test = get_metrics_torch(
        bundle,
        model,
        device=device,
    )

    return model, hist, metrics_valid, metrics_test

### **11.6. Implementación final**

In [49]:
from pathlib import Path
import json
import time
import gc
import torch

# ============================================================
# 0) Configuración final
# ============================================================
target = "delta_60"
window_size = 180
n_features = 36

batch_size_train = 4096

# HP finales MLP
hp_model = {
    "hidden_dims": [512, 256, 128],
    "activation": "relu",
    "dropout": 0.187,
    "norm": "none",
    "residual": False,
}

# HP finales optimización
hp_optim = {
    "lr": 5.13e-4,
    "weight_decay": 6.6e-5,
    "max_epochs": 30,
    "patience": 5,
}

models_dir = Path("/content/drive/MyDrive/neural_profit/final_models")
models_dir.mkdir(parents=True, exist_ok=True)

best_path = models_dir / f"mlp_L{window_size}_{target}.pt"
summary_path = best_path.with_name(best_path.stem + "_summary.json")

print("best_path:", str(best_path))
print("summary_path:", str(summary_path))


# ============================================================
# Verificar si el modelo o el resumen ya existen
# ============================================================
model_exists = best_path.exists()
summary_exists = summary_path.exists()

print("\n=== Verificación de archivos ===")
print("Modelo existe :", model_exists)
print("Resumen existe:", summary_exists)

mlp_model = None
mlp_model_summary = None

if model_exists or summary_exists:
    print("\n[WARNING] Ya existen archivos para esta configuración.")

    if model_exists:
        print("Modelo encontrado en:")
        print(best_path)
        mlp_model = torch.load(best_path, map_location=device)

    if summary_exists:
        print("Resumen encontrado en:")
        print(summary_path)
        with open(summary_path, "r", encoding="utf-8") as f:
            mlp_model_summary = json.load(f)

    print("\n[OK] Archivos cargados. Se omite el entrenamiento.")

else:
    # ============================================================
    # 1) Crear bundle
    # ============================================================
    (bundle,) = create_bundles(
        window_size=window_size,
        targets=[target],
        windows_paths=windows_paths,
        scalers_paths=scalers_paths,
        flatten_X=True,   # MLP necesita input 2D: (N, L*F)
    )

    print("Bundle creado:")
    print("train X:", bundle["train"]["X"].shape, "| y:", bundle["train"]["y"].shape)
    print("valid X:", bundle["valid"]["X"].shape, "| y:", bundle["valid"]["y"].shape)
    print("test  X:", bundle["test"]["X"].shape,  "| y:", bundle["test"]["y"].shape)

    # ============================================================
    # 2) Crear DataLoaders
    # ============================================================
    loaders = make_loaders_from_bundle(
        bundle,
        batch_size=batch_size_train,
        num_workers=0,
    )

    print("\nLoaders creados:")
    print("n_train:", len(loaders["train"].dataset))
    print("n_valid:", len(loaders["valid"].dataset))
    print("n_test :", len(loaders["test"].dataset))

    # ============================================================
    # 3) Definir input_dim real
    # ============================================================
    in_dim = bundle["train"]["X"].shape[1]
    print("\nin_dim:", in_dim)

    # ============================================================
    # 5) Entrenar MLP final + guardar checkpoint
    # ============================================================
    t0 = time.perf_counter()

    model, hist, metrics_valid, metrics_test = train_mlp(
        bundle=bundle,
        loaders=loaders,
        in_dim=in_dim,
        device=device,

        # HP finales MLP
        **hp_model,

        # HP finales optimización
        **hp_optim,

        # guardado del mejor modelo
        save_best=True,
        best_path=str(best_path),

        verbose=True,
    )

    dt_train = time.perf_counter() - t0

    # ============================================================
    # 6) Guardar JSON resumen (HP + métricas + paths)
    # ============================================================
    summary = {
        "model_name": "mlp_seq2one",
        "target": target,
        "window_size": window_size,
        "n_features": n_features,
        "in_dim": in_dim,
        "batch_size_train": batch_size_train,
        "hp_model": hp_model,
        "hp_optim": hp_optim,
        "hist": hist,
        "metrics_valid": metrics_valid,
        "metrics_test": metrics_test,
        "paths": {
            "model_path": str(best_path),
            "summary_path": str(summary_path),
        },
        "timing": {
            "train_seconds": float(dt_train),
        },
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nResumen guardado en:")
    print(str(summary_path))

    print("\n=== HIST ===")
    print(hist)

    print("\n=== METRICS VALID ===")
    print(metrics_valid)

    print("\n=== METRICS TEST ===")
    print(metrics_test)

    print("\nModelo guardado en:")
    print(str(best_path))

    # dejar variables finales con los nombres pedidos
    mlp_model = model
    mlp_model_summary = summary

    # ============================================================
    # 8) Liberar memoria
    # ============================================================
    bundle = None
    loaders = None
    model = None
    hist = None
    metrics_valid = None
    metrics_test = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n[CLEAN] Memoria liberada.")


print("\n=== OBJETOS FINALES ===")
print("mlp_model:", type(mlp_model))
print("mlp_model_summary:", type(mlp_model_summary))

best_path: /content/drive/MyDrive/neural_profit/final_models/mlp_L180_delta_60.pt
summary_path: /content/drive/MyDrive/neural_profit/final_models/mlp_L180_delta_60_summary.json

=== Verificación de archivos ===
Modelo existe : True
Resumen existe: True

[WARNING] Ya existen archivos para esta configuración.
Modelo encontrado en:
/content/drive/MyDrive/neural_profit/final_models/mlp_L180_delta_60.pt
Resumen encontrado en:
/content/drive/MyDrive/neural_profit/final_models/mlp_L180_delta_60_summary.json

[OK] Archivos cargados. Se omite el entrenamiento.

=== OBJETOS FINALES ===
mlp_model: <class 'dict'>
mlp_model_summary: <class 'dict'>


### **11.7. Conclusiones — Entrenamiento final del MLP (L = 180, target = delta_60)**

**1. Desempeño en VALID**

Las métricas obtenidas en el conjunto de validación son:

- **MAE ≈ 21.09**
- **RMSE ≈ 34.29**
- **Directional Accuracy (DA) ≈ 0.751**
- **R² ≈ 0.482**

Estos resultados muestran un desempeño sólido del modelo en validación y son **consistentes con los resultados observados durante el análisis multi-seed realizado durante el proceso de tuning**.

En particular:

- El **R² ≈ 0.48** se encuentra dentro del rango esperado según los experimentos previos.
- El **Directional Accuracy cercano a 0.75** indica que el modelo mantiene una capacidad robusta para predecir correctamente la dirección del movimiento del mercado.

---

**2. Desempeño en TEST (fuera de muestra)**

Las métricas obtenidas en el conjunto de test son:

- **MAE ≈ 34.52**
- **RMSE ≈ 58.88**
- **Directional Accuracy (DA) ≈ 0.740**
- **R² ≈ 0.492**

En el conjunto fuera de muestra el modelo mantiene:

- **Directional Accuracy elevada**
- **R² consistente con validación**
- **MAE dentro del rango esperado**

Esto indica que el modelo **generaliza adecuadamente** y que la configuración seleccionada mantiene un desempeño estable fuera de muestra.

Cabe destacar que el **R² en test es incluso ligeramente superior al observado en validación**, lo que sugiere que **no existe evidencia de sobreajuste significativo**.

---

**3. Consistencia con el proceso de tuning**

Los resultados obtenidos en el entrenamiento final son **coherentes con los resultados obtenidos durante el tuning multi-seed**.

Durante el tuning se observaron aproximadamente:

- **R²_valid_mean ≈ 0.458**
- **R²_test_mean ≈ 0.457**
- **DA_valid_mean ≈ 0.747**
- **DA_test_mean ≈ 0.728**
- **MAE_valid_mean ≈ 21.43**
- **MAE_test_mean ≈ 35.98**

El modelo final entrenado presenta métricas **muy cercanas a esos valores medios**, con ligeras mejoras esperables debido a:

- variabilidad entre semillas,
- entrenamiento completo con la configuración final,
- efecto del *early stopping* en la corrida final.

Esta consistencia confirma que **la selección de hiperparámetros realizada durante el tuning fue adecuada**.

---

**4. Confirmación de la arquitectura seleccionada**

Los resultados obtenidos confirman que:

- La **arquitectura MLP seleccionada** con capas ocultas **[512, 256, 128]** funciona correctamente para el problema de predicción intradía.
- El modelo mantiene **Directional Accuracy cercana al 74–75%**, lo que indica una buena capacidad para capturar la dirección del movimiento del mercado.
- El comportamiento observado es **consistente con los experimentos realizados durante el proceso de tuning y validación multi-seed**.

El modelo entrenado y guardado en: `/content/drive/MyDrive/neural_profit/final_models/mlp_L180_delta_60.pt
`


In [44]:
summary

{'model_name': 'mlp_seq2one',
 'target': 'delta_60',
 'window_size': 180,
 'n_features': 36,
 'in_dim': 6480,
 'batch_size_train': 4096,
 'hp_model': {'hidden_dims': [512, 256, 128],
  'activation': 'relu',
  'dropout': 0.187,
  'norm': 'none',
  'residual': False},
 'hp_optim': {'lr': 0.000513,
  'weight_decay': 6.6e-05,
  'max_epochs': 30,
  'patience': 5},
 'hist': {'train_loss': [1958.8659830485005,
   1529.7529394212054,
   1379.9126534024817,
   1274.0923882444972,
   1157.641353544323,
   1037.1409998220956,
   935.2165054467005,
   852.6040220992569,
   800.7507908099636,
   752.4279549428774,
   711.6956695943354],
  'valid_mse': [1415.3369987754172,
   1316.703332360312,
   1256.481304643447,
   1764.0322022555106,
   1630.44165432591,
   1175.9199882881471,
   1264.4189728455885,
   1647.9421883009625,
   1221.3850494104915,
   1191.034003531355,
   1258.9103295694026],
  'best_valid_mse': 1175.9199882881471,
  'best_epoch': 6,
  'epochs_ran': 11,
  'best_path': '/content/dr